<a href="https://colab.research.google.com/github/lubaochuan/ml_python/blob/main/xor_tensorflow_exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Learning XOR with TensorFlow

In this notebook, you will:

1. Build a small neural network in TensorFlow  
2. Train it to learn the XOR function  
3. Inspect the model's predictions  
4. Visualize the decision boundary  
5. Explain why a neural network can solve XOR but a single linear model cannot

## What is XOR?

The XOR function outputs 1 when the two inputs are different, and 0 when they are the same.

| x1 | x2 | XOR |
|---|---|---|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

A key idea: **XOR is not linearly separable**, so a single straight line cannot separate the two classes correctly.

## Part 1. Import libraries

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow import keras
from tensorflow.keras import layers

np.random.seed(42)
tf.random.set_seed(42)

## Part 2. Create the XOR dataset

Run the next cell and inspect the inputs and targets.

In [ ]:
X = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
], dtype=np.float32)

y = np.array([
    [0],
    [1],
    [1],
    [0]
], dtype=np.float32)

print("Inputs:")
print(X)
print("\nTargets:")
print(y)

### Questions

1. Which input pairs produce output 1?  
2. Which input pairs produce output 0?  
3. Why is this pattern difficult for a straight-line classifier?

## Part 3. Plot the training data

In [ ]:
plt.figure(figsize=(6, 6))

for i in range(len(X)):
    if y[i] == 0:
        plt.scatter(X[i, 0], X[i, 1], marker='o', s=220, label='Class 0' if i == 0 else "")
    else:
        plt.scatter(X[i, 0], X[i, 1], marker='^', s=220, label='Class 1' if i == 1 else "")

plt.xlim(-0.2, 1.2)
plt.ylim(-0.2, 1.2)
plt.xlabel("x1")
plt.ylabel("x2")
plt.title("XOR Dataset")
plt.grid(True)
plt.legend()
plt.show()

### Questions

1. Are the two class 0 points next to each other?  
2. Are the two class 1 points next to each other?  
3. Can one straight line separate them correctly?

## Part 4. Build a neural network

We will use:

- 2 input features  
- 1 hidden layer with 4 units  
- 1 output unit for binary classification  

Why this helps: the hidden layer can transform the input into a representation that makes the XOR pattern easier to separate.

In [ ]:
model = keras.Sequential([
    layers.Input(shape=(2,)),
    layers.Dense(4, activation='tanh'),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.1),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

### Questions

1. Why do we need a hidden layer for XOR?  
2. Why does the output layer use `sigmoid`?  
3. What does a sigmoid output near 1 mean?

## Part 5. Train the model

In [ ]:
history = model.fit(X, y, epochs=500, verbose=0)

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history.history['loss'])
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss")
plt.grid(True)
plt.show()

### Questions

1. Does the loss generally decrease?  
2. What would it mean if the loss stayed high?  
3. Why can a tiny dataset still need many epochs?

## Part 6. Inspect predictions

In [ ]:
predictions = model.predict(X, verbose=0)

print("Predicted probabilities:")
for i in range(len(X)):
    print(f"Input: {X[i]}  Prediction: {predictions[i][0]:.4f}  Target: {y[i][0]}")

In [ ]:
predicted_classes = (predictions > 0.5).astype(int)

print("Predicted classes:")
for i in range(len(X)):
    print(f"Input: {X[i]}  Predicted class: {predicted_classes[i][0]}  Target: {int(y[i][0])}")

### Questions

1. Which predictions are close to 1?  
2. Which predictions are close to 0?  
3. Did the model correctly learn all four XOR cases?

## Part 7. Visualize the decision boundary

We will evaluate the model on a fine grid of points to see how it divides the input space.

In [ ]:
xx, yy = np.meshgrid(
    np.linspace(-0.5, 1.5, 300),
    np.linspace(-0.5, 1.5, 300)
)

grid = np.c_[xx.ravel(), yy.ravel()]
grid_predictions = model.predict(grid, verbose=0)
zz = grid_predictions.reshape(xx.shape)

In [ ]:
plt.figure(figsize=(7, 7))
plt.contourf(xx, yy, zz, levels=50, alpha=0.8)
plt.colorbar(label="Predicted Probability of Class 1")

for i in range(len(X)):
    if y[i] == 0:
        plt.scatter(X[i, 0], X[i, 1], marker='o', s=220, edgecolor='black', label='Class 0' if i == 0 else "")
    else:
        plt.scatter(X[i, 0], X[i, 1], marker='^', s=220, edgecolor='black', label='Class 1' if i == 1 else "")

plt.xlim(-0.5, 1.5)
plt.ylim(-0.5, 1.5)
plt.xlabel("x1")
plt.ylabel("x2")
plt.title("Decision Boundary for XOR")
plt.legend()
plt.grid(True)
plt.show()

### Questions

1. Is the decision boundary a straight line?  
2. Why does XOR need a nonlinear boundary?  
3. What does the background color represent?

## Part 8. Look inside the hidden layer

This cell shows the hidden-layer activations for each XOR input.

In [ ]:
# Define an input tensor that matches the model's expected input shape
input_tensor = keras.Input(shape=(2,))

# Pass this input tensor through the first (hidden) layer of the trained model
hidden_output_tensor = model.layers[0](input_tensor)

# Create a new model that maps the input tensor to the hidden layer's output
hidden_model = keras.Model(inputs=input_tensor, outputs=hidden_output_tensor)

hidden_outputs = hidden_model.predict(X, verbose=0)

print("Hidden layer activations:")
print(hidden_outputs)

### Questions

1. Do all four inputs produce the same hidden representation?  
2. Why is it useful for the hidden layer to transform the inputs?  
3. How is the hidden layer helping the final layer classify XOR?

## Part 9. Experiment: try a model without a hidden layer

Replace the network with a single sigmoid unit and train again.

In [ ]:
linear_model = keras.Sequential([
    layers.Input(shape=(2,)),
    layers.Dense(1, activation='sigmoid')
])

linear_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.1),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

linear_history = linear_model.fit(X, y, epochs=500, verbose=0)

linear_predictions = linear_model.predict(X, verbose=0)
print("Linear model probabilities:")
for i in range(len(X)):
    print(f"Input: {X[i]}  Prediction: {linear_predictions[i][0]:.4f}  Target: {y[i][0]}")

In [ ]:
linear_model = keras.Sequential([
    layers.Input(shape=(2,)),
    layers.Dense(2, activation='sigmoid'),
    layers.Dense(1, activation='sigmoid')
])

linear_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.1),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

linear_history = linear_model.fit(X, y, epochs=500, verbose=0)

linear_predictions = linear_model.predict(X, verbose=0)
print("Linear model probabilities:")
for i in range(len(X)):
    print(f"Input: {X[i]}  Prediction: {linear_predictions[i][0]:.4f}  Target: {y[i][0]}")

### Questions

1. Can the simpler model learn XOR perfectly?  
2. Why not?  
3. What does this show about linear models?

## Part 10. Optional extensions

Try one change at a time:

- Change the hidden layer size from 4 to 2 or 8  (you can duplicate the previous code cell and add `layers.Dense(2, activation='sigmoid')` to `keras.Sequential()`)
- Replace `tanh` with `relu`  
- Change the learning rate from `0.1` to `0.01` or `0.5`  
- Train on a denser XOR-style dataset

Record what happens to:
- training loss
- predictions
- decision boundary
- stability of training

## Takeaways

- XOR is a classic nonlinear classification problem.  
- A single linear classifier cannot solve XOR because the classes are not linearly separable.  
- A hidden layer lets the network transform the data into a more useful internal representation.  
- The output layer then separates that transformed representation.  
- The decision-boundary plot helps students connect the geometry to the model's behavior.